# Análisis Exploratorio
Melisa Mendizabal - Belen Monterroso - Renato Rojas

## Carga, armonización y calidad de datos

In [ ]:
import os
import pandas as pd
from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .appName("Lab7_ENEIC")
    .master("local[*]")
    .config("spark.driver.host", "127.0.0.1")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

mydir = "/opt/app/working_dir/lab7/"

COLUMNAS_REQUERIDAS = [
    "NUM_HOGAR", "NUM_PERSONA", "FACTOR", "ANIO", "TRIMESTRE",
    "OCUPADOS", "P05C16", "P05D01",
    "P02A03", "P05C07A", "P05C07B", "P05H01A",
    "P03A03A", "DOMINIO",
]

# archivo: (nombre_excel, periodo_archivo, anio_archivo, trimestre_calendario)
ARCHIVOS_2025 = {
    "2025T1": ("personas_2025_t1.xlsx", 2025, 1),
    "2025T2": ("personas_2025_t2.xlsx", 2025, 2),
    "2025T3": ("personas_2025_t3.xlsx", 2025, 3),
    "2025T4": ("personas_2025_t4.xlsx", 2025, 4),
}
ARCHIVO_2026 = {"2026T1": ("personas_2026_t1.xlsx", 2026, 1)}

def excel_a_parquet(nombre_excel, periodo_archivo, anio_archivo, trimestre_cal):
    ruta_in = os.path.join(mydir, nombre_excel)
    df_pd = pd.read_excel(ruta_in)
    df_pd.columns = [c.strip().upper() for c in df_pd.columns]
    df_pd = df_pd[[c for c in COLUMNAS_REQUERIDAS if c in df_pd.columns]]

    # Homologar: un mismo código puede llegar como número o texto -> forzar a texto
    for c in ["P05C16", "P03A03A", "DOMINIO", "OCUPADOS"]:
        if c in df_pd.columns:
            df_pd[c] = df_pd[c].apply(lambda v: None if pd.isna(v) else str(v).strip())

    for c in ["NUM_HOGAR", "NUM_PERSONA", "ANIO", "TRIMESTRE"]:
        if c in df_pd.columns:
            df_pd[c] = pd.to_numeric(df_pd[c], errors="coerce")

    for c in ["P02A03", "P05C07A", "P05C07B", "P05H01A", "P05D01", "FACTOR"]:
        if c in df_pd.columns:
            df_pd[c] = pd.to_numeric(df_pd[c], errors="coerce")

    df_pd["archivo_origen"] = nombre_excel
    df_pd["periodo_archivo"] = periodo_archivo
    df_pd["anio_archivo"] = anio_archivo
    df_pd["trimestre_calendario"] = trimestre_cal

    ruta_out = os.path.join(mydir, "raw_parquet", f"{periodo_archivo}.parquet")
    os.makedirs(os.path.dirname(ruta_out), exist_ok=True)
    df_pd.to_parquet(ruta_out, index=False)
    return ruta_out